# Event-based analysis: comparison and country plots

This notebook:
1. Runs `identify_events` on the $H = P/X$ index from `data_analysis_functions.py`
2. Runs the old reciprocal $R2NER = X/P$ event detection for comparison
3. Compares which events are detected by each approach
4. Plots country-by-country $H$ time series with shaded devaluation events
5. Plots country-by-country $R2NER$ time series with shaded events (old approach)
6. Runs event detection on the nominal exchange rate $1/X$ (USD per LCU) and plots the grid

In [1]:
%run ../source/data_analysis_functions.py
hybrid = build_hybrid_index_and_ma_df(start="1980-01-01", end="2023-12-31")
hybrid.head()

No data for STP
No data for SSD


,country,year,fx_index,baseline,nominal_fx
4,AGO,2006,190.712770,139.401448,0.012442
5,AGO,2007,239.141575,167.229763,0.013036
6,AGO,2008,278.785009,200.701820,0.013328
7,AGO,2009,294.147693,232.070651,0.012606
8,AGO,2010,286.127866,257.782983,0.010881


## 1. Detect events on the H index

In [2]:
# Run identify_events on the H index
events_H, hybrid_events = identify_events(
    hybrid.copy(), 
    trigger_threshold=0.1, 
    end_threshold=0.15
)
print(f"Events detected on H index: {len(events_H)}")
print(f"Countries with events: {events_H['Country'].nunique()}")
events_H.head(10)

Events detected on H index: 104
Countries with events: 50


,Country,Start_Date,End_Date,Peak_Date,Duration,Peak_Devaluation,Devaluation_at_End,Drawdown_Area
0,DZA,1990,2004,2000,15,-29.940603,-13.024037,-313.015898
1,DZA,2014,2017,2016,4,-13.814566,-9.496171,-34.360295
2,AGO,2017,2022,2020,6,-44.189220,-8.471357,-137.042709
3,BEN,1984,1986,1985,3,-26.929902,-6.688078,-55.376874
4,BEN,1993,1995,1994,3,-28.716765,-8.544002,-37.260767
5,BEN,2014,2017,2016,4,-12.739092,-10.144773,-34.907606
6,BWA,1996,2000,1998,5,-19.095588,-10.468403,-72.949520
7,BWA,1984,1987,1985,4,-30.667448,-9.657362,-68.091304
8,BWA,2014,2017,2016,4,-16.390814,-9.137594,-43.826253
9,BFA,1993,2003,2000,11,-26.193547,-1.906001,-275.124583


## 2. Detect events on the reciprocal R2NER index (old approach)

In [3]:
# Replicate the old notebook's event detection on the reciprocal index R2NER = X/P
from scipy.signal import argrelextrema

def detect_events_reciprocal(start_date="2000", ma_window=5, 
                              rise_threshold=0.1, return_threshold=0.1):
    """
    Detect events using the OLD reciprocal index R2NER = X/P.
    This mirrors the events_From_baseline notebook logic.
    Returns both the events DataFrame and the R2NER time series for plotting.
    """
    sd = pd.to_datetime(start_date)
    sd_minus = sd - pd.DateOffset(years=ma_window-1)
    df = data.copy()
    df.index = pd.to_datetime(df.index)
    df = df[(df.index >= sd_minus) & (df.index <= "2023-12-31")]
    
    results_old = []
    ts_records = []  # time series for plotting
    
    for country in african_countries:
        panusatls_col = f"{country}PANUSATLS"
        price_level_col = f"{country}NECONPRVTXN"
        
        if panusatls_col not in df.columns or price_level_col not in df.columns:
            continue
        
        # R2NER = X/P (reciprocal of H = P/X)
        r2ner = df[panusatls_col] / df[price_level_col]
        first_valid = r2ner.first_valid_index()
        if first_valid is None:
            continue
        
        r2ner_norm = r2ner / r2ner.loc[first_valid] * 100
        r2ner_ma = r2ner_norm.rolling(window=ma_window, min_periods=ma_window).mean()
        
        # Store time series for plotting
        for date, val, ma_val in zip(r2ner_norm.index, r2ner_norm.values, r2ner_ma.values):
            if date >= sd and not pd.isna(val) and not pd.isna(ma_val):
                ts_records.append({
                    "country": country,
                    "year": date.year,
                    "r2ner": float(val),
                    "r2ner_baseline": float(ma_val),
                })
        
        # Filter to analysis period
        mask = r2ner_norm.index >= sd
        values = r2ner_norm[mask].values
        times = r2ner_norm[mask].index
        ma_5year = r2ner_ma[mask].values
        
        if len(values) < 3 or np.all(np.isnan(ma_5year)):
            continue
        
        # Peak detection on values (rising = devaluation in reciprocal)
        peak_indices = argrelextrema(values, np.greater)[0]
        if len(values) >= 2 and values[-1] > values[-2]:
            peak_indices = np.append(peak_indices, len(values) - 1)
        
        country_events = []
        
        for t_idx in np.arange(len(times)):
            if np.isnan(ma_5year[t_idx]):
                continue
            possible_peaks = peak_indices[peak_indices > t_idx]
            if possible_peaks.size == 0:
                continue
            
            for p_idx in possible_peaks:
                increase = values[p_idx] / ma_5year[t_idx] - 1.0
                if increase >= rise_threshold:
                    threshold = ma_5year[t_idx] * (1. + return_threshold)
                    
                    # Check for clean rise
                    intermediate_troughs = 0
                    for i in range(t_idx+1, p_idx):
                        if values[i] < threshold:
                            intermediate_troughs += 1
                    if intermediate_troughs > 0:
                        continue
                    
                    fall_date = None
                    for idx in range(p_idx + 1, len(values)):
                        if values[idx] < threshold:
                            fall_date = times[idx]
                            max_peak = p_idx
                            for i in peak_indices[(peak_indices > p_idx) & (peak_indices < idx)]:
                                if values[i] > values[max_peak]:
                                    max_peak = i
                            gain_at_peak = (values[max_peak] - ma_5year[t_idx]) / ma_5year[t_idx] * 100
                            integrand = np.maximum(values/ma_5year[t_idx] - 1, 0.0) * 100
                            area = np.sum(integrand[t_idx:idx+1])
                            break
                    else:
                        idx = len(values) - 1
                        fall_date = times[idx]
                        max_peak = p_idx
                        for i in peak_indices[peak_indices > p_idx]:
                            if values[i] > values[max_peak]:
                                max_peak = i
                        gain_at_peak = (values[max_peak] - ma_5year[t_idx]) / ma_5year[t_idx] * 100
                        integrand = np.maximum(values/ma_5year[t_idx] - 1, 0.0) * 100
                        area = np.sum(integrand[t_idx:idx+1])
                    
                    country_events.append({
                        "Country": country,
                        "Start_Date": times[t_idx].year,
                        "End_Date": fall_date.year if hasattr(fall_date, 'year') else pd.Timestamp(fall_date).year,
                        "Peak_Date": times[max_peak].year,
                        "Peak_Devaluation": gain_at_peak,
                        "Drawdown_Area": area,
                        "Duration": (fall_date.year if hasattr(fall_date, 'year') else pd.Timestamp(fall_date).year) - times[t_idx].year + 1
                    })
        
        # Filter overlapping (same logic as old notebook: sort by area descending)
        if country_events:
            country_events.sort(key=lambda x: x['Drawdown_Area'], reverse=True)
            selected = []
            used_intervals = []
            for event in country_events:
                start = event['Start_Date']
                end = event['End_Date']
                overlap = False
                for s, e in used_intervals:
                    if not (end < s or start > e):
                        overlap = True
                        break
                if not overlap:
                    selected.append(event)
                    used_intervals.append((start, end))
            results_old.extend(selected)
    
    events_df = pd.DataFrame(results_old)
    ts_df = pd.DataFrame(ts_records).sort_values(["country", "year"]).reset_index(drop=True)
    return events_df, ts_df

events_R2NER, r2ner_df = detect_events_reciprocal(start_date="1980", rise_threshold=0.1, return_threshold=0.15)
print(f"Events detected on reciprocal R2NER index: {len(events_R2NER)}")
print(f"Countries with events: {events_R2NER['Country'].nunique()}")
events_R2NER.head(10)

Events detected on reciprocal R2NER index: 118
Countries with events: 51


,Country,Start_Date,End_Date,Peak_Date,Peak_Devaluation,Drawdown_Area,Duration
0,DZA,1990,1993,1991,36.386061,77.973373,4
1,DZA,2015,2017,2016,14.828629,35.437405,3
2,DZA,1994,1996,1995,12.225582,25.311124,3
3,DZA,1999,2001,2000,12.738887,22.622299,3
4,AGO,2018,2022,2020,74.709822,201.189993,5
5,AGO,2015,2017,2016,10.353017,10.353017,3
6,BEN,1984,1986,1985,33.496269,62.707075,3
7,BEN,1993,1995,1994,39.516465,48.259306,3
8,BEN,2015,2017,2016,13.156391,35.282003,3
9,BWA,1984,1987,1985,44.039872,91.729705,4


## 3. Compare the two event sets

The two approaches use mirrored indices ($H = P/X$ falls during devaluation, $R2NER = X/P$ rises).
Small differences are expected due to Jensen's inequality: $\text{MA}(1/x) \neq 1/\text{MA}(x)$,
which causes the moving-average baselines to differ slightly between the two formulations.

In [4]:
# Compare the two sets of events
comparison = events_H[['Country', 'Start_Date', 'End_Date', 'Peak_Date', 'Peak_Devaluation', 'Drawdown_Area']].merge(
    events_R2NER[['Country', 'Start_Date', 'End_Date', 'Peak_Date', 'Peak_Devaluation', 'Drawdown_Area']],
    on=['Country', 'Start_Date'],
    how='outer',
    suffixes=('_H', '_R2NER'),
    indicator=True
)

print("=== EVENT COMPARISON: H index vs R2NER (reciprocal) ===")
print(f"\nEvents in H only:      {(comparison['_merge'] == 'left_only').sum()}")
print(f"Events in R2NER only:  {(comparison['_merge'] == 'right_only').sum()}")
print(f"Events in both:        {(comparison['_merge'] == 'both').sum()}")

print("\n--- Events matched on Country + Start_Date ---")
matched = comparison[comparison['_merge'] == 'both'].copy()
if len(matched) > 0:
    matched['end_match'] = matched['End_Date_H'] == matched['End_Date_R2NER']
    matched['peak_match'] = matched['Peak_Date_H'] == matched['Peak_Date_R2NER']
    print(f"End date matches:  {matched['end_match'].sum()} / {len(matched)}")
    print(f"Peak date matches: {matched['peak_match'].sum()} / {len(matched)}")

print("\n--- Events only in H index ---")
h_only = comparison[comparison['_merge'] == 'left_only'][['Country', 'Start_Date', 'End_Date_H', 'Peak_Devaluation_H']]
if len(h_only) > 0:
    print(h_only.to_string(index=False))
else:
    print("None")

print("\n--- Events only in R2NER (reciprocal) ---")
r_only = comparison[comparison['_merge'] == 'right_only'][['Country', 'Start_Date', 'End_Date_R2NER', 'Peak_Devaluation_R2NER']]
if len(r_only) > 0:
    print(r_only.to_string(index=False))
else:
    print("None")

print("\n--- Matched events with different end dates ---")
if len(matched) > 0:
    mismatched_ends = matched[~matched['end_match']][['Country', 'Start_Date', 'End_Date_H', 'End_Date_R2NER']]
    if len(mismatched_ends) > 0:
        print(mismatched_ends.to_string(index=False))
    else:
        print("All end dates match")

=== EVENT COMPARISON: H index vs R2NER (reciprocal) ===



Events in H only:      25
Events in R2NER only:  39
Events in both:        79

--- Events matched on Country + Start_Date ---
End date matches:  62 / 79
Peak date matches: 61 / 79

--- Events only in H index ---
Country  Start_Date  End_Date_H  Peak_Devaluation_H
    AGO        2017      2022.0          -44.189220
    BEN        2014      2017.0          -12.739092
    BWA        2014      2017.0          -16.390814
    COD        2002      2004.0          -10.419824
    DZA        2014      2017.0          -13.814566
    GAB        1996      2003.0          -39.382658
    GMB        1998      2006.0          -54.460813
    GMB        2012      2016.0          -19.265955
    MAR        1999      2002.0          -17.858327
    MAR        2014      2017.0          -12.700088
    MDG        1998      2000.0          -11.548143
    MLI        1996      1999.0          -16.512688
    MOZ        1990      2023.0          -49.658970
    MRT        1997      2005.0          -30.435732
    NA

## 4. Country-by-country plots of $H$ with shaded devaluation events

In [5]:
import altair as alt

charts = []

for country in sorted(hybrid_events['country'].unique()):
    cdata = hybrid_events[hybrid_events['country'] == country].copy()
    if len(cdata) == 0:
        continue
    
    # Base line chart: H index and baseline
    base_data = cdata.melt(
        id_vars=['year'],
        value_vars=['fx_index', 'baseline'],
        var_name='series',
        value_name='value'
    )
    
    line = alt.Chart(base_data).mark_line().encode(
        x=alt.X('year:O', title='Year', axis=alt.Axis(labelAngle=-45, values=list(range(1980, 2024, 5)))),
        y=alt.Y('value:Q', title='H index (P/X, normalized to 100)'),
        color=alt.Color('series:N', 
                        scale=alt.Scale(domain=['fx_index', 'baseline'], range=['steelblue', 'orange']),
                        legend=alt.Legend(title=None, orient='bottom')),
        strokeDash=alt.StrokeDash('series:N',
                                   scale=alt.Scale(domain=['fx_index', 'baseline'], range=[[], [5, 5]]),
                                   legend=None)
    )
    
    # Shaded event regions
    country_events = events_H[events_H['Country'] == country]
    rects = []
    annotations = []
    for _, ev in country_events.iterrows():
        rect_data = pd.DataFrame([{
            'start': ev['Start_Date'], 
            'end': ev['End_Date'],
        }])
        rect = alt.Chart(rect_data).mark_rect(
            opacity=0.15, color='red'
        ).encode(
            x='start:O',
            x2='end:O'
        )
        rects.append(rect)
        
        # Annotation with peak devaluation
        ann_data = pd.DataFrame([{
            'year': ev['Peak_Date'],
            'label': f"{ev['Peak_Devaluation']:.0f}%",
            'y_pos': cdata['fx_index'].min() * 0.95
        }])
        ann = alt.Chart(ann_data).mark_text(
            color='red', fontSize=9, fontWeight='bold', dy=-5
        ).encode(
            x='year:O',
            y='y_pos:Q',
            text='label:N'
        )
        annotations.append(ann)
    
    chart = line
    for r in rects:
        chart = chart + r
    for a in annotations:
        chart = chart + a
    
    chart = chart.properties(
        title=country,
        width=280,
        height=160
    )
    charts.append(chart)

# Arrange in a grid (4 columns)
n_cols = 4
rows = []
for i in range(0, len(charts), n_cols):
    row_charts = charts[i:i+n_cols]
    if len(row_charts) == 1:
        rows.append(row_charts[0])
    else:
        rows.append(alt.hconcat(*row_charts))

grid = alt.vconcat(*rows).resolve_scale(
    y='independent'
).properties(
    title='H index (P/X) with devaluation events shaded'
)

grid

alt.VConcatChart(...)

## 5. Paired country-by-country plots: $H = P/X$ vs $R2NER = X/P$

For each country, the **top** panel shows the $H$ index (falls during devaluation)
with events from `identify_events`, and the **bottom** panel shows the reciprocal
$R2NER$ index (rises during devaluation) with events from the old detection approach.
Countries are arranged four per row for easy visual comparison.

In [6]:
# Paired grid: H (top) and R2NER (bottom) per country, 4 countries per row
country_pairs = []

# Get the union of countries present in both datasets
countries_h = set(hybrid_events['country'].unique())
countries_r = set(r2ner_df['country'].unique())
all_countries = sorted(countries_h | countries_r)

for country in all_countries:
    panels = []

    # --- Top panel: H index ---
    cdata_h = hybrid_events[hybrid_events['country'] == country].copy()
    if len(cdata_h) > 0:
        h_melt = cdata_h.melt(
            id_vars=['year'], value_vars=['fx_index', 'baseline'],
            var_name='series', value_name='value'
        )
        h_line = alt.Chart(h_melt).mark_line().encode(
            x=alt.X('year:O', title=None,
                    axis=alt.Axis(labelAngle=-45, values=list(range(1980, 2024, 10)))),
            y=alt.Y('value:Q', title='H = P/X'),
            color=alt.Color('series:N',
                            scale=alt.Scale(domain=['fx_index', 'baseline'],
                                            range=['steelblue', 'orange']),
                            legend=None),
            strokeDash=alt.StrokeDash('series:N',
                                      scale=alt.Scale(domain=['fx_index', 'baseline'],
                                                      range=[[], [5, 5]]),
                                      legend=None),
        )
        h_layers = [h_line]
        for _, ev in events_H[events_H['Country'] == country].iterrows():
            rd = pd.DataFrame([{'start': ev['Start_Date'], 'end': ev['End_Date']}])
            h_layers.append(
                alt.Chart(rd).mark_rect(opacity=0.15, color='red').encode(
                    x='start:O', x2='end:O')
            )
            ann = pd.DataFrame([{
                'year': ev['Peak_Date'],
                'label': f"{ev['Peak_Devaluation']:.0f}%",
                'y_pos': cdata_h['fx_index'].min() * 0.95
            }])
            h_layers.append(
                alt.Chart(ann).mark_text(color='red', fontSize=8, fontWeight='bold', dy=-5
                ).encode(x='year:O', y='y_pos:Q', text='label:N')
            )
        h_chart = alt.layer(*h_layers).properties(
            title=alt.Title(country, anchor='middle'),
            width=250, height=120
        )
        panels.append(h_chart)

    # --- Bottom panel: R2NER ---
    cdata_r = r2ner_df[r2ner_df['country'] == country].copy()
    if len(cdata_r) > 0:
        r_melt = cdata_r.melt(
            id_vars=['year'], value_vars=['r2ner', 'r2ner_baseline'],
            var_name='series', value_name='value'
        )
        r_line = alt.Chart(r_melt).mark_line().encode(
            x=alt.X('year:O', title='Year',
                    axis=alt.Axis(labelAngle=-45, values=list(range(1980, 2024, 10)))),
            y=alt.Y('value:Q', title='R2NER = X/P'),
            color=alt.Color('series:N',
                            scale=alt.Scale(domain=['r2ner', 'r2ner_baseline'],
                                            range=['steelblue', 'orange']),
                            legend=None),
            strokeDash=alt.StrokeDash('series:N',
                                      scale=alt.Scale(domain=['r2ner', 'r2ner_baseline'],
                                                      range=[[], [5, 5]]),
                                      legend=None),
        )
        r_layers = [r_line]
        for _, ev in events_R2NER[events_R2NER['Country'] == country].iterrows():
            rd = pd.DataFrame([{'start': ev['Start_Date'], 'end': ev['End_Date']}])
            r_layers.append(
                alt.Chart(rd).mark_rect(opacity=0.15, color='red').encode(
                    x='start:O', x2='end:O')
            )
            ann = pd.DataFrame([{
                'year': ev['Peak_Date'],
                'label': f"+{ev['Peak_Devaluation']:.0f}%",
                'y_pos': cdata_r['r2ner'].max() * 1.05
            }])
            r_layers.append(
                alt.Chart(ann).mark_text(color='red', fontSize=8, fontWeight='bold', dy=5
                ).encode(x='year:O', y='y_pos:Q', text='label:N')
            )
        r_chart = alt.layer(*r_layers).properties(
            width=250, height=120
        )
        panels.append(r_chart)

    if panels:
        pair = alt.vconcat(*panels).resolve_scale(y='independent')
        country_pairs.append(pair)

# Arrange in rows of 4 country pairs
n_cols = 4
rows = []
for i in range(0, len(country_pairs), n_cols):
    row = country_pairs[i:i+n_cols]
    if len(row) == 1:
        rows.append(row[0])
    else:
        rows.append(alt.hconcat(*row))

grid_paired = alt.vconcat(*rows).properties(
    title='H = P/X (top, falls in devaluation) vs R2NER = X/P (bottom, rises in devaluation)'
)

grid_paired

alt.VConcatChart(...)

## 6. Nominal exchange rate $1/X$ (USD per LCU): event detection and plots

Here we run the same `identify_events` algorithm on the **nominal** exchange rate $1/X_{i,t}$ (USD per LCU),
using its own 5-year moving average as baseline. A fall in $1/X$ is a nominal devaluation.
This isolates the pure nominal FX moves, without the price-level adjustment present in $H$.

In [7]:
# Build nominal FX tidy dataframe: 1/X normalized to 100 with its own 5-year MA baseline
ma_window = 5
start_date = pd.to_datetime("1980-01-01")
sd_minus = start_date - pd.DateOffset(years=ma_window - 1)

df_raw = data.copy()
df_raw.index = pd.to_datetime(df_raw.index)
df_raw = df_raw[(df_raw.index >= sd_minus) & (df_raw.index <= "2023-12-31")]

nom_records = []
for country in african_countries:
    panusatls_col = f"{country}PANUSATLS"
    if panusatls_col not in df_raw.columns:
        continue
    
    # 1/X = USD per LCU (falls during devaluation)
    nom_fx = 1.0 / df_raw[panusatls_col]
    first_valid = nom_fx.first_valid_index()
    if first_valid is None:
        continue
    
    nom_norm = nom_fx / nom_fx.loc[first_valid] * 100
    nom_ma = nom_norm.rolling(window=ma_window, min_periods=ma_window).mean()
    
    for date, val, ma_val in zip(nom_norm.index, nom_norm.values, nom_ma.values):
        if date >= start_date and not pd.isna(val) and not pd.isna(ma_val):
            nom_records.append({
                "country": country,
                "year": date.year,
                "fx_index": float(val),      # identify_events expects this column name
                "baseline": float(ma_val),    # identify_events expects this column name
            })

nominal_df = pd.DataFrame(nom_records).sort_values(["country", "year"]).reset_index(drop=True)

# Run identify_events on the nominal FX series
events_nom, nominal_events = identify_events(
    nominal_df.copy(),
    trigger_threshold=0.1,
    end_threshold=0.1
)
print(f"Events detected on nominal 1/X: {len(events_nom)}")
print(f"Countries with events: {events_nom['Country'].nunique()}")
events_nom.head(10)

Events detected on nominal 1/X: 69
Countries with events: 51


,Country,Start_Date,End_Date,Peak_Date,Duration,Peak_Devaluation,Devaluation_at_End,Drawdown_Area
0,DZA,1987,2023,2022,37,-96.572131,-96.416889,-3138.326357
1,DZA,1984,1986,1985,3,-11.171519,-5.016624,-26.568238
2,AGO,1990,2023,2023,34,-100.000000,-100.000000,-3299.748879
3,BEN,1993,2023,2022,31,-54.614848,-53.328646,-1458.600650
4,BEN,1984,1987,1985,4,-32.009615,1.634926,-73.900455
5,BWA,1984,2023,2023,40,-92.703782,-92.703782,-2987.430658
6,BFA,1993,2023,2022,31,-54.614847,-53.328646,-1458.600648
7,BFA,1984,1987,1985,4,-32.009615,1.634926,-73.900455
8,BDI,1984,2023,2023,40,-96.299655,-96.299655,-3011.467879
9,CMR,1993,2023,2022,31,-54.614847,-53.328646,-1458.600648


In [8]:
# Country-by-country plot of nominal 1/X with shaded devaluation events
charts_nom = []

for country in sorted(nominal_events['country'].unique()):
    cdata = nominal_events[nominal_events['country'] == country].copy()
    if len(cdata) == 0:
        continue
    
    base_data = cdata.melt(
        id_vars=['year'],
        value_vars=['fx_index', 'baseline'],
        var_name='series',
        value_name='value'
    )
    
    line = alt.Chart(base_data).mark_line().encode(
        x=alt.X('year:O', title='Year', axis=alt.Axis(labelAngle=-45, values=list(range(1980, 2024, 5)))),
        y=alt.Y('value:Q', title='Nominal 1/X (USD per LCU, norm. 100)'),
        color=alt.Color('series:N',
                        scale=alt.Scale(domain=['fx_index', 'baseline'], range=['steelblue', 'orange']),
                        legend=alt.Legend(title=None, orient='bottom')),
        strokeDash=alt.StrokeDash('series:N',
                                   scale=alt.Scale(domain=['fx_index', 'baseline'], range=[[], [5, 5]]),
                                   legend=None)
    )
    
    # Shaded event regions
    country_ev = events_nom[events_nom['Country'] == country]
    rects = []
    annotations = []
    for _, ev in country_ev.iterrows():
        rect_data = pd.DataFrame([{
            'start': ev['Start_Date'],
            'end': ev['End_Date'],
        }])
        rect = alt.Chart(rect_data).mark_rect(
            opacity=0.15, color='red'
        ).encode(
            x='start:O',
            x2='end:O'
        )
        rects.append(rect)
        
        ann_data = pd.DataFrame([{
            'year': ev['Peak_Date'],
            'label': f"{ev['Peak_Devaluation']:.0f}%",
            'y_pos': cdata['fx_index'].min() * 0.95
        }])
        ann = alt.Chart(ann_data).mark_text(
            color='red', fontSize=9, fontWeight='bold', dy=-5
        ).encode(
            x='year:O',
            y='y_pos:Q',
            text='label:N'
        )
        annotations.append(ann)
    
    chart = line
    for r in rects:
        chart = chart + r
    for a in annotations:
        chart = chart + a
    
    chart = chart.properties(
        title=country,
        width=280,
        height=160
    )
    charts_nom.append(chart)

# Arrange in a grid (4 columns)
n_cols = 4
rows_nom = []
for i in range(0, len(charts_nom), n_cols):
    row_charts = charts_nom[i:i+n_cols]
    if len(row_charts) == 1:
        rows_nom.append(row_charts[0])
    else:
        rows_nom.append(alt.hconcat(*row_charts))

grid_nom = alt.vconcat(*rows_nom).resolve_scale(
    y='independent'
).properties(
    title='Nominal exchange rate 1/X (USD per LCU) with devaluation events shaded'
)

grid_nom

alt.VConcatChart(...)

## 7. Dual-axis grid: $ (left) vs nominal /X$ (right) with events

Green shading marks 569Xlindex devaluation events (real); red shading marks nominal /X$ events.
Where only green appears, inflation offset the nominal devaluation enough that $ did not trigger.
Where only red appears, a nominal devaluation occurred but was absorbed by rising prices.

In [9]:
# Dual-axis grid: H (left y, green) and 1/X (right y, red) with respective events
# Merge H and nominal data on country + year
combined = hybrid_events[['country', 'year', 'fx_index', 'baseline']].merge(
    nominal_events[['country', 'year', 'fx_index', 'baseline']],
    on=['country', 'year'],
    how='inner',
    suffixes=('_H', '_nom')
)

charts_dual = []

for country in sorted(combined['country'].unique()):
    cdata = combined[combined['country'] == country].copy()
    if len(cdata) == 0:
        continue

    # --- Left axis: H index (green tones) ---
    h_data = cdata.melt(
        id_vars=['year'],
        value_vars=['fx_index_H', 'baseline_H'],
        var_name='series',
        value_name='value_H'
    )
    left = alt.Chart(h_data).mark_line().encode(
        x=alt.X('year:O', title='Year', axis=alt.Axis(labelAngle=-45, values=list(range(1980, 2024, 5)))),
        y=alt.Y('value_H:Q', title='H = P/X (left)'),
        color=alt.Color('series:N',
                        scale=alt.Scale(domain=['fx_index_H', 'baseline_H'], range=['green', 'darkgreen']),
                        legend=alt.Legend(title=None, orient='bottom',
                                         labelExpr="datum.label == 'fx_index_H' ? 'H index' : 'H baseline'")),
        strokeDash=alt.StrokeDash('series:N',
                                   scale=alt.Scale(domain=['fx_index_H', 'baseline_H'], range=[[], [5, 5]]),
                                   legend=None)
    )

    # --- Right axis: nominal 1/X (blue tones) ---
    nom_data = cdata.melt(
        id_vars=['year'],
        value_vars=['fx_index_nom', 'baseline_nom'],
        var_name='series',
        value_name='value_nom'
    )
    right = alt.Chart(nom_data).mark_line().encode(
        x=alt.X('year:O'),
        y=alt.Y('value_nom:Q', title='1/X nominal (right)', axis=alt.Axis(orient='right')),
        color=alt.Color('series:N',
                        scale=alt.Scale(domain=['fx_index_nom', 'baseline_nom'], range=['steelblue', 'navy']),
                        legend=alt.Legend(title=None, orient='bottom',
                                         labelExpr="datum.label == 'fx_index_nom' ? '1/X nominal' : '1/X baseline'")),
        strokeDash=alt.StrokeDash('series:N',
                                   scale=alt.Scale(domain=['fx_index_nom', 'baseline_nom'], range=[[], [5, 5]]),
                                   legend=None)
    )

    # --- Green rects for H events ---
    layers = [left, right]
    for _, ev in events_H[events_H['Country'] == country].iterrows():
        rd = pd.DataFrame([{'start': ev['Start_Date'], 'end': ev['End_Date']}])
        layers.append(
            alt.Chart(rd).mark_rect(opacity=0.12, color='green').encode(x='start:O', x2='end:O')
        )

    # --- Red rects for nominal events ---
    for _, ev in events_nom[events_nom['Country'] == country].iterrows():
        rd = pd.DataFrame([{'start': ev['Start_Date'], 'end': ev['End_Date']}])
        layers.append(
            alt.Chart(rd).mark_rect(opacity=0.12, color='red').encode(x='start:O', x2='end:O')
        )

    chart = alt.layer(*layers).resolve_scale(
        y='independent',
        color='independent'
    ).properties(
        title=country,
        width=320,
        height=180
    )
    charts_dual.append(chart)

# Arrange in a grid (3 columns — wider panels)
n_cols = 3
rows_dual = []
for i in range(0, len(charts_dual), n_cols):
    row_charts = charts_dual[i:i+n_cols]
    if len(row_charts) == 1:
        rows_dual.append(row_charts[0])
    else:
        rows_dual.append(alt.hconcat(*row_charts))

grid_dual = alt.vconcat(*rows_dual).properties(
    title='H index (green, left axis) vs nominal 1/X (blue, right axis) — events: green = H, red = nominal'
)

grid_dual

alt.VConcatChart(...)

## 8. Comparative analysis: $H$ vs nominal $1/X$ events from 1990

We re-run event detection from 1990 on both indices, **excluding country-years under
fixed exchange rate regimes** (Somalia, Liberia, Djibouti throughout; Zimbabwe 2009-2018
during dollarisation; Angola 2004-2014 under a managed peg).  Under a fixed rate the
nominal rate cannot devalue, so including those years would inflate the denominator and
artificially depress nominal prevalence.

In [10]:
# Rebuild both indices from 1990 for maximum sample
hybrid_90 = build_hybrid_index_and_ma_df(start='1990-01-01', end='2023-12-31')

# Build nominal 1/X from 1990
ma_window_90 = 5
sd_90 = pd.to_datetime('1990-01-01')
sd_minus_90 = sd_90 - pd.DateOffset(years=ma_window_90 - 1)
df_raw_90 = data.copy()
df_raw_90.index = pd.to_datetime(df_raw_90.index)
df_raw_90 = df_raw_90[(df_raw_90.index >= sd_minus_90) & (df_raw_90.index <= '2023-12-31')]

nom_records_90 = []
for country in african_countries:
    col = f'{country}PANUSATLS'
    if col not in df_raw_90.columns:
        continue
    nom_fx = 1.0 / df_raw_90[col]
    fv = nom_fx.first_valid_index()
    if fv is None:
        continue
    nom_norm = nom_fx / nom_fx.loc[fv] * 100
    nom_ma = nom_norm.rolling(window=ma_window_90, min_periods=ma_window_90).mean()
    for date, val, ma_val in zip(nom_norm.index, nom_norm.values, nom_ma.values):
        if date >= sd_90 and not pd.isna(val) and not pd.isna(ma_val):
            nom_records_90.append({
                'country': country, 'year': date.year,
                'fx_index': float(val), 'baseline': float(ma_val),
            })
nominal_90 = pd.DataFrame(nom_records_90).sort_values(['country','year']).reset_index(drop=True)

# --- Exclude fixed exchange rate regime country-years ---
fixed_regime_years = {
    'SOM': list(range(1990, 2024)),  # pegged to USD throughout sample
    'LBR': list(range(1990, 2024)),  # pegged to USD throughout sample
    'DJI': list(range(1990, 2024)),  # pegged to USD throughout sample
    'ZWE': list(range(2009, 2019)),  # dollarisation 2009-2018
    'AGO': list(range(2004, 2015)),  # managed peg 2004-2014 (X ≈ 83-98)
}

fixed_cy = {(c, y) for c, yrs in fixed_regime_years.items() for y in yrs}

mask_h = hybrid_90.apply(lambda r: (r['country'], r['year']) not in fixed_cy, axis=1)
mask_n = nominal_90.apply(lambda r: (r['country'], r['year']) not in fixed_cy, axis=1)
hybrid_90_flex = hybrid_90[mask_h].copy()
nominal_90_flex = nominal_90[mask_n].copy()

n_excl_h = len(hybrid_90) - len(hybrid_90_flex)
n_excl_n = len(nominal_90) - len(nominal_90_flex)
print(f'Excluded fixed-regime country-years:  H = {n_excl_h}  |  nominal = {n_excl_n}')

# Run identify_events on filtered data
ev_H_90, df_H_90 = identify_events(hybrid_90_flex.copy(), trigger_threshold=0.1, end_threshold=0.1)
ev_nom_90, df_nom_90 = identify_events(nominal_90_flex.copy(), trigger_threshold=0.1, end_threshold=0.1)

print(f'H events from 1990 (excl. fixed): {len(ev_H_90)}  |  Nominal events: {len(ev_nom_90)}')
print(f'H countries: {ev_H_90["Country"].nunique()}  |  Nominal countries: {ev_nom_90["Country"].nunique()}')

No data for STP
No data for SSD
Excluded fixed-regime country-years:  H = 59  |  nominal = 119


H events from 1990 (excl. fixed): 92  |  Nominal events: 57
H countries: 47  |  Nominal countries: 51


In [11]:
# Build comparative statistics
def event_stats(events, df_events, label):
    total_cy = len(df_events)
    event_cy = int(df_events['in_event'].sum())
    prevalence = event_cy / total_cy if total_cy > 0 else 0
    n_events = len(events)
    mean_dur = events['Duration'].mean() if n_events > 0 else 0
    mean_peak = events['Peak_Devaluation'].mean() if n_events > 0 else 0
    mean_area = events['Drawdown_Area'].mean() if n_events > 0 else 0
    # Annual severity: mean |annual_loss| when in event
    event_years = df_events[df_events['in_event'] == 1]
    mean_annual_sev = event_years['annual_loss'].abs().mean() * 100 if len(event_years) > 0 else 0
    # Pure risk premium = prevalence * severity
    premium = prevalence * mean_annual_sev
    return {
        'Index': label,
        'Total events': n_events,
        'Countries affected': events['Country'].nunique() if n_events > 0 else 0,
        'Total country-years': total_cy,
        'Country-years in event': event_cy,
        'Prevalence (%)': round(prevalence * 100, 1),
        'Mean duration (yrs)': round(mean_dur, 1),
        'Mean peak deviation (%)': round(mean_peak, 1),
        'Mean drawdown area (pp-yrs)': round(mean_area, 1),
        'Mean annual severity (%)': round(mean_annual_sev, 2),
        'Pure risk premium (%)': round(premium, 2),
    }

stats_H = event_stats(ev_H_90, df_H_90, 'H = P/X (real)')
stats_nom = event_stats(ev_nom_90, df_nom_90, '1/X (nominal)')

comp_df = pd.DataFrame([stats_H, stats_nom]).set_index('Index').T
comp_df['Ratio nom/real'] = ''

# Compute ratios for numeric rows
for idx in comp_df.index:
    h_val = comp_df.loc[idx, 'H = P/X (real)']
    n_val = comp_df.loc[idx, '1/X (nominal)']
    if isinstance(h_val, (int, float)) and h_val != 0:
        comp_df.loc[idx, 'Ratio nom/real'] = round(n_val / h_val, 2)

print('=== COMPARATIVE EVENT STATISTICS (1990-2023) ===\n')
print(comp_df.to_string())

print('\n--- Interpretation ---')
n_ev = stats_nom['Total events']
h_ev = stats_H['Total events']
print(f'Nominal approach detects {n_ev} events vs {h_ev} on H ({n_ev/h_ev:.1f}x more).')
print(f'Nominal prevalence is {stats_nom["Prevalence (%)"]:.1f}% vs {stats_H["Prevalence (%)"]:.1f}% for H.')
dur_ratio = stats_nom['Mean duration (yrs)'] / stats_H['Mean duration (yrs)'] if stats_H['Mean duration (yrs)'] > 0 else 0
print(f'Nominal events last {stats_nom["Mean duration (yrs)"]:.1f} yrs vs {stats_H["Mean duration (yrs)"]:.1f} yrs for H ({dur_ratio:.1f}x).')
area_ratio = abs(stats_nom['Mean drawdown area (pp-yrs)'] / stats_H['Mean drawdown area (pp-yrs)']) if stats_H['Mean drawdown area (pp-yrs)'] != 0 else 0
print(f'Mean drawdown area is {area_ratio:.1f}x larger for nominal events.')
prem_ratio = stats_nom['Pure risk premium (%)'] / stats_H['Pure risk premium (%)'] if stats_H['Pure risk premium (%)'] != 0 else 0
print(f'Pure risk premium: {stats_nom["Pure risk premium (%)"]:.2f}% (nominal) vs {stats_H["Pure risk premium (%)"]:.2f}% (H) — nominal is {prem_ratio:.1f}x higher.')

=== COMPARATIVE EVENT STATISTICS (1990-2023) ===

Index                        H = P/X (real)  1/X (nominal) Ratio nom/real
Total events                          92.00          57.00           0.62
Countries affected                    47.00          51.00           1.09
Total country-years                 1504.00        1661.00            1.1
Country-years in event               702.00        1561.00           2.22
Prevalence (%)                        46.70          94.00           2.01
Mean duration (yrs)                    7.60          27.60           3.63
Mean peak deviation (%)              -28.20         -72.00           2.55
Mean drawdown area (pp-yrs)         -176.40       -1793.60          10.17
Mean annual severity (%)              23.12          65.49           2.83
Pure risk premium (%)                 10.79          61.55            5.7

--- Interpretation ---
Nominal approach detects 57 events vs 92 on H (0.6x more).
Nominal prevalence is 94.0% vs 46.7% for H.
Nominal e

In [12]:
# Per-country comparison: drawdown area nominal vs H
# Aggregate by country: total drawdown area and number of events
h_by_c = ev_H_90.groupby('Country').agg(
    n_events_H=('Duration', 'count'),
    total_area_H=('Drawdown_Area', 'sum'),
    total_duration_H=('Duration', 'sum'),
).reset_index()

nom_by_c = ev_nom_90.groupby('Country').agg(
    n_events_nom=('Duration', 'count'),
    total_area_nom=('Drawdown_Area', 'sum'),
    total_duration_nom=('Duration', 'sum'),
).reset_index()

scatter_data = h_by_c.merge(nom_by_c, on='Country', how='outer').fillna(0)
scatter_data['total_area_H'] = scatter_data['total_area_H'].abs()
scatter_data['total_area_nom'] = scatter_data['total_area_nom'].abs()

scatter_data['regime'] = scatter_data['Country'].map(
    lambda c: fx_regime.get(c, 'Flexible')
)

max_val = max(scatter_data['total_area_H'].max(), scatter_data['total_area_nom'].max()) * 1.1

scatter = alt.Chart(scatter_data).mark_circle(size=60).encode(
    x=alt.X('total_area_H:Q', title='Total drawdown area — H = P/X (real, pp-yrs)',
            scale=alt.Scale(domain=[0, max_val])),
    y=alt.Y('total_area_nom:Q', title='Total drawdown area — 1/X (nominal, pp-yrs)',
            scale=alt.Scale(domain=[0, max_val])),
    color=alt.Color('regime:N', title='FX regime',
                    scale=alt.Scale(domain=['EUR-linked', 'ZAR-linked', 'Flexible'],
                                    range=['#1f77b4', '#ff7f0e', '#2ca02c'])),
    tooltip=['Country:N', 'regime:N',
             alt.Tooltip('total_area_H:Q', format='.0f', title='H drawdown'),
             alt.Tooltip('total_area_nom:Q', format='.0f', title='Nominal drawdown'),
             'n_events_H:Q', 'n_events_nom:Q'],
).properties(
    title='Total drawdown area by country: nominal vs real (1990-2023)',
    width=450, height=450
)

line_data = pd.DataFrame({'x': [0, max_val], 'y': [0, max_val]})
diag = alt.Chart(line_data).mark_line(strokeDash=[5,5], color='grey').encode(
    x='x:Q', y='y:Q'
)

# Filter to outliers for labels
threshold = max_val * 0.3
outliers = scatter_data[
    (scatter_data['total_area_nom'] > threshold) | (scatter_data['total_area_H'] > threshold)
].copy()
labels = alt.Chart(outliers).mark_text(
    align='left', dx=7, fontSize=9
).encode(
    x='total_area_H:Q',
    y='total_area_nom:Q',
    text='Country:N'
)

(scatter + diag + labels).properties(
    title='Cumulative event severity: nominal vs real (points above diagonal = nominal worse)'
)

alt.LayerChart(...)

In [13]:
# Duration comparison: total country-years spent in events
dur_scatter = alt.Chart(scatter_data).mark_circle(size=60).encode(
    x=alt.X('total_duration_H:Q', title='Total years in H events'),
    y=alt.Y('total_duration_nom:Q', title='Total years in nominal events'),
    color=alt.Color('regime:N', title='FX regime',
                    scale=alt.Scale(domain=['EUR-linked', 'ZAR-linked', 'Flexible'],
                                    range=['#1f77b4', '#ff7f0e', '#2ca02c'])),
    tooltip=['Country:N', 'regime:N', 'total_duration_H:Q', 'total_duration_nom:Q'],
).properties(width=400, height=400)

max_dur = max(scatter_data['total_duration_H'].max(), scatter_data['total_duration_nom'].max()) * 1.1
diag_dur = alt.Chart(pd.DataFrame({'x': [0, max_dur], 'y': [0, max_dur]})).mark_line(
    strokeDash=[5,5], color='grey'
).encode(x='x:Q', y='y:Q')

# Filter to outliers for labels
dur_threshold = max_dur * 0.4
dur_outliers = scatter_data[
    (scatter_data['total_duration_nom'] > dur_threshold) | (scatter_data['total_duration_H'] > dur_threshold)
].copy()
labels_dur = alt.Chart(dur_outliers).mark_text(
    align='left', dx=7, fontSize=9
).encode(
    x='total_duration_H:Q',
    y='total_duration_nom:Q',
    text='Country:N'
)

(dur_scatter + diag_dur + labels_dur).properties(
    title='Total years in event: nominal vs real (above diagonal = nominal longer)'
)

alt.LayerChart(...)

In [14]:
# Summary bar chart of key metrics side by side
metrics = [
    ('Number of events', stats_H['Total events'], stats_nom['Total events']),
    ('Prevalence (%)', stats_H['Prevalence (%)'], stats_nom['Prevalence (%)']),
    ('Mean duration (yrs)', stats_H['Mean duration (yrs)'], stats_nom['Mean duration (yrs)']),
    ('Mean annual severity (%)', stats_H['Mean annual severity (%)'], stats_nom['Mean annual severity (%)']),
    ('Pure risk premium (%)', stats_H['Pure risk premium (%)'], stats_nom['Pure risk premium (%)']),
]

bar_records = []
for name, h_val, n_val in metrics:
    bar_records.append({'Metric': name, 'Index': 'H = P/X (real)', 'Value': h_val})
    bar_records.append({'Metric': name, 'Index': '1/X (nominal)', 'Value': n_val})
bar_df = pd.DataFrame(bar_records)

bars = alt.Chart(bar_df).mark_bar().encode(
    x=alt.X('Index:N', title=None, axis=alt.Axis(labelAngle=0)),
    y=alt.Y('Value:Q', title=None),
    color=alt.Color('Index:N',
                    scale=alt.Scale(domain=['H = P/X (real)', '1/X (nominal)'],
                                    range=['green', 'indianred']),
                    legend=None),
    column=alt.Column('Metric:N', title=None,
                      sort=[m[0] for m in metrics]),
).properties(width=120, height=250)

bars

alt.Chart(...)

## 9. Kenya drill-down: $H$, nominal $1/X$, and $1/P$

Plot the Kenyan $H = P/X$ on the left y-axis, the nominal exchange rate $1/X$
and the inverse price level $1/P$ on the right y-axis.  Since $H = P/X$, a fall in $H$
can be decomposed into a fall in $1/X$ (nominal devaluation) partially offset by a
fall in $1/P$ (inflation).

To ensure visual comparability, $1/P$ is renormalised so that its starting value
equals $w$ times the starting value of $1/X$, where $w$ is the ratio of the maximum
percentage-point amplitude of $1/P$ to that of $1/X$ (both measured from a common
base of 100).

In [15]:
# Kenya drill-down: H (left axis) vs 1/X and 1/P (right axis)
country = 'KEN'

# --- Retrieve H and nominal 1/X from already-built dataframes ---
ken_H = hybrid_events[hybrid_events['country'] == country][['year', 'fx_index']].copy()
ken_H.columns = ['year', 'H']

ken_nom = nominal_events[nominal_events['country'] == country][['year', 'fx_index']].copy()
ken_nom.columns = ['year', 'inv_X']

# --- Build 1/P, normalised to 100 at the start of the sample ---
import pandas as pd, numpy as np
df_raw_k = data.copy()
df_raw_k.index = pd.to_datetime(df_raw_k.index)
price_col = f'{country}NECONPRVTXN'
P = df_raw_k[price_col].dropna()
inv_P = 1.0 / P

first_year = ken_nom['year'].min()
inv_P_start = inv_P.loc[inv_P.index.year == first_year]
inv_P_start_val = inv_P_start.iloc[0] if len(inv_P_start) > 0 else inv_P.iloc[0]
inv_P_norm = inv_P / inv_P_start_val * 100   # starts at 100
ken_invP = pd.DataFrame({'year': inv_P_norm.index.year, 'inv_P_raw': inv_P_norm.values})
ken_invP = ken_invP[ken_invP['year'].between(ken_nom['year'].min(), ken_nom['year'].max())]

# --- Merge ---
ken = ken_H.merge(ken_nom, on='year', how='outer').merge(ken_invP, on='year', how='outer').sort_values('year')

# --- Amplitude-based renormalisation of 1/P ---
# Normalise both to 100 at start for amplitude comparison
inv_X_start = ken['inv_X'].dropna().iloc[0]
inv_X_norm100 = ken['inv_X'] / inv_X_start * 100
A_X = 100 - inv_X_norm100.min()        # max pp drop from 100
A_P = 100 - ken['inv_P_raw'].min()      # max pp drop from 100 (already at 100)

w = A_P / A_X if A_X > 0 else 1.0
new_start = w * inv_X_start
ken['inv_P'] = ken['inv_P_raw'] / 100 * new_start

print(f'Max amplitude (from base 100):  1/X = {A_X:.1f} pp  |  1/P = {A_P:.1f} pp')
print(f'w = {w:.3f}  |  1/X start: {inv_X_start:.1f}  |  1/P renormalised start: {new_start:.1f}')

# --- Left axis: H index (no baseline) ---
left = alt.Chart(ken).mark_line(strokeWidth=2, color='green').encode(
    x=alt.X('year:O', title='Year', axis=alt.Axis(labelAngle=-45,
            values=list(range(int(ken['year'].min()), int(ken['year'].max())+1, 5)))),
    y=alt.Y('H:Q', title='H = P/X  (left axis)'),
)

# --- Right axis: 1/X and 1/P (renormalised), both solid ---
r_data = ken.melt(id_vars='year', value_vars=['inv_X', 'inv_P'],
                  var_name='series', value_name='val_R')
right = alt.Chart(r_data).mark_line(strokeWidth=2).encode(
    x=alt.X('year:O'),
    y=alt.Y('val_R:Q', title='1/X  and  1/P  (right axis)',
            axis=alt.Axis(orient='right')),
    color=alt.Color('series:N',
                    scale=alt.Scale(domain=['inv_X', 'inv_P'],
                                    range=['steelblue', 'firebrick']),
                    legend=alt.Legend(title=None, orient='bottom-right',
                                     labelExpr="datum.label == 'inv_X' ? '1/X (nominal FX)' : '1/P (inverse price, rescaled)'")),
)

# --- H event shading (green) ---
layers = [left, right]
for _, ev in events_H[events_H['Country'] == country].iterrows():
    rd = pd.DataFrame([{'start': ev['Start_Date'], 'end': ev['End_Date']}])
    layers.append(
        alt.Chart(rd).mark_rect(opacity=0.12, color='green').encode(x='start:O', x2='end:O')
    )

# --- Nominal event shading (red) ---
for _, ev in events_nom[events_nom['Country'] == country].iterrows():
    rd = pd.DataFrame([{'start': ev['Start_Date'], 'end': ev['End_Date']}])
    layers.append(
        alt.Chart(rd).mark_rect(opacity=0.12, color='red').encode(x='start:O', x2='end:O')
    )

chart_ken = alt.layer(*layers).resolve_scale(
    y='independent', color='independent'
).properties(
    title='Kenya — H = P/X (green, left) vs 1/X (blue) and 1/P rescaled (red) on right axis',
    width=700, height=350
)

chart_ken

Max amplitude (from base 100):  1/X = 89.7 pp  |  1/P = 97.6 pp
w = 1.089  |  1/X start: 51.5  |  1/P renormalised start: 56.0


alt.LayerChart(...)